# 01 - CasADi Symbolic Framework

This notebook introduces the core symbolic objects from CasADi documentation Section 3. The goal is to become comfortable with symbolic matrices before using them in QPs and NLPs.

Learning goals:

- Create and inspect `SX`, `MX`, and `DM` matrices.
- Understand shapes, sparsity, indexing, slicing, and concatenation.
- Distinguish elementwise multiplication `*` from matrix multiplication `@`.
- Compute Jacobians, gradients, Hessians, and directional derivatives.

In [ ]:
# Colab setup.
# These tutorials assume a fresh Google Colab runtime.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "casadi",
    "numpy",
    "matplotlib",

])


In [ ]:
import casadi as ca
import numpy as np

print("CasADi version:", ca.__version__)

## SX Symbols

`SX` is CasADi's scalar-expression symbolic type. An `SX` matrix is a matrix of scalar symbolic expressions.

In [ ]:
x = ca.SX.sym("x")
y = ca.SX.sym("y", 5)
Z = ca.SX.sym("Z", 3, 2)

print("x =", x, "shape", x.shape)
print("y =", y, "shape", y.shape)
print("Z =", Z, "shape", Z.shape)

In [ ]:
expr_scalar = ca.sqrt(x**2 + 10)
expr_vector = ca.sin(y) + y**2
expr_matrix = Z**2 + 2 * Z + 1

print("scalar expression:", expr_scalar)
print("vector expression:", expr_vector)
print("matrix expression:\n", expr_matrix)

## DM Numeric Matrices

`DM` stores numeric matrices. You will often evaluate symbolic `Function` objects with `DM` or NumPy data.

In [ ]:
A = ca.DM([[1, 2, 0], [0, 3, 4]])
b = ca.DM([1, 2, 3])

print("A =\n", A)
print("b =", b)
print("A @ b =", A @ b)
print("as NumPy array:\n", np.array(A))

## Sparse And Dense Matrices

CasADi stores a sparsity pattern separately from expression values. This matters later for solvers.

In [ ]:
dense_zero = ca.SX.zeros(3, 3)
sparse_zero = ca.SX(3, 3)
identity = ca.SX.eye(3)

lower_pattern = ca.Sparsity.lower(3)
L = ca.SX.sym("L", lower_pattern)

print("dense_zero sparsity:", dense_zero.sparsity())
print("sparse_zero sparsity:", sparse_zero.sparsity())
print("identity sparsity:", identity.sparsity())
print("lower-triangular symbolic matrix:\n", L)
print("lower pattern:", L.sparsity())

## Indexing, Slicing, And Concatenation

CasADi indexing is close to NumPy indexing, but the result is still a symbolic matrix.

In [ ]:
M = ca.SX.sym("M", 3, 3)

first_column = M[:, 0]
upper_left = M[0:2, 0:2]
stacked = ca.vertcat(first_column, ca.SX([10, 20, 30]))
wide = ca.horzcat(M, ca.SX.eye(3))

print("first column:", first_column)
print("upper-left block:\n", upper_left)
print("vertical concatenation shape:", stacked.shape)
print("horizontal concatenation shape:", wide.shape)

## Elementwise `*` Versus Matrix `@`

In CasADi Python, `*` is elementwise multiplication. Use `@` for matrix multiplication.

In [ ]:
A_num = ca.DM([[1, 2], [3, 4]])
B_num = ca.DM([[2, 0], [0, 2]])

print("A * B =\n", A_num * B_num)
print("A @ B =\n", A_num @ B_num)

## SX Versus MX

`SX` expands operations scalar-by-scalar. `MX` keeps larger graph operations, which is useful for composing functions and solvers.

In [ ]:
X_sx = ca.SX.sym("X", 2, 2)
y_sx = ca.SX.sym("y")
f_sx = 3 * X_sx * X_sx + y_sx

X_mx = ca.MX.sym("X", 2, 2)
y_mx = ca.MX.sym("y")
f_mx = 3 * X_mx * X_mx + y_mx

print("SX expression:\n", f_sx)
print("MX expression:\n", f_mx)

Do not mix `SX` and `MX` directly in one expression graph. If you need both, wrap the `SX` expression in a `Function` and call that function from an `MX` graph.

In [ ]:
try:
    bad = X_sx + X_mx
except TypeError as err:
    print("Expected error when mixing SX and MX directly:")
    print(err)

sx_fun = ca.Function("sx_fun", [X_sx, y_sx], [f_sx])
composed_mx = sx_fun(ca.DM.eye(2), 1.0) + X_mx
print("MX graph containing a call to an SX-defined Function:\n", composed_mx)

## Automatic Differentiation

CasADi differentiates symbolic expressions exactly by graph transformation.

In [ ]:
q = ca.SX.sym("q", 2)
residual = ca.vertcat(
    q[0] + 2 * q[1] - 1,
    ca.sin(q[0]) - q[1]
)
cost = 0.5 * ca.sumsqr(residual)

J = ca.jacobian(residual, q)
grad = ca.gradient(cost, q)
H, grad_from_hessian = ca.hessian(cost, q)

print(cost)
print(J)
print(grad)
print(grad_from_hessian)
print(H)

## Exercise

Build a residual for fitting a line to three data points. The model is

$$
\hat y_i(a, b) = a x_i + b.
$$

Stack the decision variables as

$$
\theta = \begin{bmatrix} a \\ b \end{bmatrix}.
$$

The residual vector is

$$
r(\theta) =
\begin{bmatrix}
a x_1 + b - y_1 \\
a x_2 + b - y_2 \\
a x_3 + b - y_3
\end{bmatrix}.
$$

The least-squares problem is

$$
\begin{aligned}
\min_{\theta \in \mathbb{R}^2} \quad
& \frac{1}{2} \|r(\theta)\|_2^2.
\end{aligned}
$$

Compute

$$
J_r(\theta) = \frac{\partial r}{\partial \theta}, \qquad
\nabla f(\theta) = \frac{\partial f}{\partial \theta}, \qquad
\nabla^2 f(\theta) = \frac{\partial^2 f}{\partial \theta^2}.
$$